# 01 Exploratory Data Analysis
Understand the data before building any model. This phase shows that we think like engineers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## Load and inspect the data

In [ ]:
# Column names for CMAPSS dataset
cols = ['unit', 'cycle', 'setting1', 'setting2', 'setting3'] + [f's{i}' for i in range(1, 22)]

train = pd.read_csv('../data/raw/train_FD001.txt', sep=' ', header=None, names=cols)
train.dropna(axis=1, inplace=True) # Remove empty columns

print(train.shape) # Should be ~20,000 rows
print(train.head())
print(train.describe())

## Create the Remaining Useful Life (RUL) target column

In [ ]:
# RUL = max cycle for each engine - current cycle
max_cycles = train.groupby('unit')['cycle'].max().reset_index()
max_cycles.columns = ['unit', 'max_cycle']
train = train.merge(max_cycles, on='unit')
train['RUL'] = train['max_cycle'] - train['cycle']
train.drop('max_cycle', axis=1, inplace=True)

print('RUL column created. Sample:', train['RUL'].describe())

## Visualize sensor readings

In [ ]:
# Plot sensor trends for one engine over its lifetime
engine_1 = train[train['unit'] == 1]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
sensors = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13']
for ax, sensor in zip(axes.flat, sensors):
    ax.plot(engine_1['cycle'], engine_1[sensor])
    ax.set_title(sensor)
    ax.set_xlabel('Cycle')
plt.suptitle('Sensor readings over engine lifetime', fontsize=14)
plt.tight_layout()
plt.show()